In [6]:
import re
from collections import Counter

def clean_text(text):
    text = text.lower()
    text = re.sub(r"'s\b", "", text)      
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def save_clean_corpus(lines, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        for line in lines:
            clean = clean_text(line)
            if clean:
                f.write(clean + "\n")

def load_corpus(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()
    return lines


def tokenize(lines):
    tokens = []
    for line in lines:
        clean = clean_text(line)
        tokens.extend(clean.split())
    return tokens

def corpus_stats(tokens):
    total_tokens = len(tokens)
    vocab = set(tokens)
    vocab_size = len(vocab)

    counter = Counter(tokens)

    print("=== Corpus Stats ===")
    print(f"Total tokens: {total_tokens}")
    print(f"Vocabulary size: {vocab_size}")
    print(f"Top 20 words: {counter.most_common(20)}")

    return counter

file_path = "gaming_corpus_raw.txt"

lines = load_corpus(file_path)
tokens = tokenize(lines)
counter = corpus_stats(tokens)
save_clean_corpus(lines, "gaming_corpus_clean.txt")

=== Corpus Stats ===
Total tokens: 141147
Vocabulary size: 10978
Top 20 words: [('the', 6006), ('game', 4734), ('i', 3471), ('and', 3254), ('to', 3110), ('you', 2887), ('a', 2815), ('it', 2557), ('is', 2339), ('this', 2219), ('of', 2213), ('factory', 1453), ('must', 1394), ('in', 1315), ('grow', 1268), ('for', 1255), ('good', 1252), ('that', 1042), ('but', 972), ('my', 906)]


In [4]:
import requests
import time

def get_reviews(appid, max_reviews=5000):
    url = f"https://store.steampowered.com/appreviews/{appid}"
    reviews = []
    cursor = "*"
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    while len(reviews) < max_reviews:
        params = {
            "json": 1,
            "language": "english",
            "filter": "recent",
            "num_per_page": 100,
            "cursor": cursor
        }

        try:
            r = requests.get(url, params=params, headers=headers, timeout=10)
            data = r.json()
        except Exception as e:
            print("Error:", e)
            time.sleep(5)
            continue

        for review in data["reviews"]:
            text = review["review"]
            reviews.append(text)

        cursor = data["cursor"]

        if not data["reviews"]:
            break

        time.sleep(1)

    return reviews[:max_reviews]

dota = get_reviews(570, 3000)
csgo = get_reviews(730, 3000)
factorio = get_reviews(427520, 3000)

all_reviews = dota + csgo + factorio

with open("gaming_corpus_raw.txt", "w", encoding="utf-8") as f:
    for r in all_reviews:
        f.write(r.replace("\n", " ") + "\n")

print("Done:", len(all_reviews))

Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read timed out. (read timeout=10)
Error: HTTPSConnectionPool(host='store.steampowered.com', port=443): Read

In [11]:
import requests
from bs4 import BeautifulSoup
import time
import re

HEADERS = {"User-Agent": "Mozilla/5.0"}


# === Safe request с retry ===
def safe_get(url, retries=5, timeout=10):
    for i in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=timeout)
            r.raise_for_status()
            return r
        except requests.exceptions.RequestException as e:
            print(f"[Retry {i+1}] Error:", e)
            time.sleep(2)

    return None


# === 1. Очистка текста ===
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# === 2. Получить список тем ===
def get_discussion_threads(appid, pages=2):
    threads = []

    for page in range(pages):
        url = f"https://steamcommunity.com/app/{appid}/discussions/?p={page+1}"
        r = safe_get(url)

        if r is None:
            continue

        soup = BeautifulSoup(r.text, "html.parser")

        links = soup.select(".forum_topic_name")

        for link in links:
            a_tag = link.find("a")

            if not a_tag:
                continue

            href = a_tag.get("href")

            if href and href.startswith("http"):
                title = a_tag.text.strip()
                threads.append((title, href))

        print(f"Collected threads from page {page+1}")
        time.sleep(2)

    return threads


# === 3. Парсинг одной темы ===
def parse_thread(thread_url):
    if not thread_url:
        return []

    r = safe_get(thread_url)

    if r is None:
        return []

    soup = BeautifulSoup(r.text, "html.parser")

    posts = []

    comments = soup.select(".commentthread_comment_text")

    for c in comments:
        text = c.text.strip()
        clean = clean_text(text)

        if len(clean.split()) > 3:
            posts.append(clean)

    return posts


# === 4. Сбор корпуса ===
def build_corpus(appid, max_threads=20):
    corpus = []

    threads = get_discussion_threads(appid, pages=2)

    if not threads:
        print("No threads found.")
        return corpus

    for i, (title, url) in enumerate(threads[:max_threads]):
        print(f"Parsing thread {i+1}/{len(threads)}")

        try:
            thread_texts = parse_thread(url)
            corpus.extend(thread_texts)
        except Exception as e:
            print("Error:", e)

        time.sleep(2)

    return corpus


# === 5. Сохранение корпуса ===
def save_corpus(corpus, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for line in corpus:
            f.write(line + "\n")


# === 6. Запуск ===
if __name__ == "__main__":
    appid = 570  # Dota 2

    corpus = build_corpus(appid, max_threads=30)

    print("Total documents:", len(corpus))

    save_corpus(corpus, "steam_discussions_dota2.txt")

Collected threads from page 1
Collected threads from page 2
No threads found.
Total documents: 0


In [ ]:
import nltk
nltk.download('brown')
from nltk.corpus import brown

tokens = [w.lower() for w in brown.words() if w.isalpha()]
base_tokens = tokens[:140000]


[nltk_data] Downloading package brown to
[nltk_data]     C:\Users\k4ty2\AppData\Roaming\nltk_data...
[nltk_data]   Package brown is already up-to-date!


['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', 'recent', 'primary', 'election', 'produced', 'no', 'evidence', 'that', 'any', 'irregularities', 'took', 'place', 'the', 'jury', 'further', 'said', 'in', 'presentments', 'that', 'the', 'city', 'executive', 'committee', 'which', 'had', 'charge', 'of', 'the', 'election', 'deserves', 'the', 'praise', 'and', 'thanks', 'of', 'the', 'city', 'of', 'atlanta', 'for', 'the', 'manner', 'in', 'which', 'the', 'election', 'was', 'conducted', 'the', 'term', 'jury', 'had', 'been', 'charged', 'by', 'fulton', 'superior', 'court', 'judge', 'durwood', 'pye', 'to', 'investigate', 'reports', 'of', 'possible', 'irregularities', 'in', 'the', 'primary', 'which', 'was', 'won', 'by', 'ivan', 'allen', 'only', 'a', 'relative', 'handful', 'of', 'such', 'reports', 'was', 'received', 'the', 'jury', 'said', 'considering', 'the', 'widespread']
